# ENGR 418 Project Stage 2


## Group 5


Jason Schultz - 24662652


Matthew Hinchliff - 97352314


## Results of new dataset on old code


Using an image width of 64:


Confusion Matrix:
[[17  7  3]
 [ 3 22  2]
 [10  6 11]]

Accuracy Score:
0.6172839506172839

In [ ]:
'''
Sets up all library dependencies as well all required function calls
to easily fit the classifiers to a lego shape data set.
'''

# Import necessary libraries for data processing and model training
import os
import numpy as np
from numpy import ndarray
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix)
from PIL import Image, ImageFilter
from scipy.ndimage import rotate

        
def get_matching_image_files(folder: str, label: str) -> list[str]:
    '''
    Reads in all images at a given folder path and returns all files
    that contain the label string in it.
    '''
    # Get the list of all files in the folder
    file_names = os.listdir(folder)
    
    # Filter files based on the label prefix
    label_files = []
    for file in file_names:
        if file.startswith(label):
            label_files.append(file)
    
    return label_files

        
def read_image_as_array(im_width: int, filepath: str) -> ndarray:
    '''
    Takes in a path to an image and resizes and returns it to a square 
    image based on given im_width value.
    '''
    im = Image.open(filepath).convert("L")  # Convert to grayscale
    im = im.resize((im_width, im_width))  # Resize to target width
    im_array = np.asarray(im)  # Convert image to array

    return im_array


def get_classifier_data(
        folder: str,
        im_width: int,
        label: str,
        label_index: int,
) -> tuple[ndarray, ndarray]:
    ''' 
    Reads in image data for a single classifier and returns its feature array
    and label array
    '''
    label_files = get_matching_image_files(folder, label)

    # Initialize empty arrays for storing image data and labels
    x = np.empty((len(label_files), im_width**2))  # Features
    y = np.empty((len(label_files), 1))            # Labels

    # Read images, resize, and store them in x; assign corresponding label to y
    for i in range(len(label_files)):
        filepath = folder + label_files[i]
        im_array = read_image_as_array(im_width, filepath)
        x[i:] = im_array.reshape(1, -1)  # Flatten image array
        y[i, 0] = label_index  # Assign label index to the output array

    return x, y  # Return feature and label arrays


def get_feature_and_label_arrays(
        path: str,
        labels: list[str],
        im_width: int,
) -> tuple[ndarray, ndarray]:
    ''' 
    Obtains the feature matrix and label vector for all classifiers and concatenates
    them all to a single array, x and y.
    '''
    # Initialize lists to store the training data
    x_list = []
    y_list = []

    # Loop through each label to load data, using the get_data function defined earlier
    for (i, label) in enumerate(labels):
        x_i, y_i = get_classifier_data(path, im_width, label, i)  # Get data for each label
        x_list.append(x_i)  # Add features (x) to the list
        y_list.append(y_i)  # Add labels (y) to the list

    # Initialize x_train and y_train with the first label's data
    x = np.array(x_list[0])
    y = np.array(y_list[0])

    # Append data from the remaining labels to x_train and y_train
    for i in range(1, len(x_list)):
        x = np.append(x, x_list[i], axis=0)
        y = np.append(y, y_list[i], axis=0)

    return x, y


def test_function(
        path: str,
        model: LogisticRegression,
        labels: list[str],
        im_width: int,
) -> None:
    ''' 
    Predicts the classifiers of a given data set in path for the model and
    prints out the confusion matrix and accuracy score
    '''
    x, y = get_feature_and_label_arrays(path, labels, im_width)
    y_pred = model.predict(x)
    evaluate_model_metrics(y, y_pred)



def evaluate_model_metrics(true_labels, predicted_labels) -> None:
    ''' 
    Function to print the confusion matrix and accuracy score to evaluate model performance
    '''
    print("Confusion Matrix:")
    print(confusion_matrix(true_labels, predicted_labels))
    print("\nAccuracy Score:")
    print(accuracy_score(true_labels, predicted_labels))

def process_image(image_array, edge_thresh, peak_threshold, im_width, im_length):
    image_array = (image_array - np.min(image_array))*255/(np.max(image_array) - np.min(image_array))
    im = Image.fromarray(image_array.reshape(im_width,im_length)).convert('L')

    edges_image =  im.filter(ImageFilter.FIND_EDGES)
    edges_array = np.asarray(edges_image).astype(float)

    # Remove weird boundary around image (just 2 lines below)
    edges_array_scaled = edges_array.copy()[1:im_width-1,1:im_length-1]
    edges_array_scaled[edges_array_scaled < edge_thresh] = 0

    best_angle_v = 0
    best_angle_h = 0
    max_sum_v = 0
    max_sum_h = 0

    M = 5
    f = np.insert(np.zeros(M-1),M-1,1,axis=0) - (1/M)*np.ones(M)

    for angle in range(0, 90):
        rotated_array = rotate(edges_array_scaled, angle, reshape=False, mode='nearest')

        edges_v = np.sum(rotated_array, axis=0)
        edges_h = np.sum(rotated_array, axis=1)

        convolved_v = np.convolve(edges_v, f, mode='valid')
        convolved_h = np.convolve(edges_h, f, mode='valid')

        peak_v = np.max(np.abs(convolved_v))
        peak_h = np.max(np.abs(convolved_h))

        if peak_v > max_sum_v:
            max_sum_v = peak_v
            best_angle_v = angle
        if peak_h > max_sum_h:
            max_sum_h = peak_h
            best_angle_h = angle

    #best_angle = best_angle_h if max_sum_h > max_sum_v else best_angle_v
    best_angle = best_angle_v
    edges_array_rotated = rotate(edges_array_scaled, best_angle)

    # plt.figure(figsize=(10, 5))
    # plt.subplot(1, 2, 1)
    # plt.title("Original Image")
    # plt.imshow(image_array.reshape(im_width, im_length), cmap='gray')

    # plt.subplot(1, 2, 2)
    # plt.title(f"Rotated Image (Best Angle: {best_angle_v}°)")
    # plt.imshow(edges_array_rotated, cmap='gray')
    # plt.show()

    edges_v = np.sum(edges_array_rotated, axis=0)
    edges_h = np.sum(edges_array_rotated, axis=1)
    
    edges_v = np.convolve(edges_v, f, mode='valid')
    edges_h = np.convolve(edges_h, f, mode='valid')
    
    peak_v = np.max(np.abs(edges_v))
    peak_h = np.max(np.abs(edges_h))
    sd_v = np.sqrt(np.var(edges_v[np.abs(edges_v)< peak_threshold*peak_v]))
    sd_h = np.sqrt(np.var(edges_h[np.abs(edges_h)< peak_threshold*peak_h]))
    
    x = np.array([peak_v/sd_v, peak_h/sd_h])
    #ratio = (peak_v/sd_v)/ (peak_h/sd_h)

    #features = np.concatenate((x, [ratio]))

    return x

In [54]:
'''
Training cell

This cell sets up the required variables to run the test as well as fits
the model to the training data set. A test is also performed to see its
accuracy on the trained data.
'''
# Initializing required parameters
path_training = "../project/Lego_dataset_2/training/"
path_testing = "../project/Lego_dataset_2/testing/"
im_width = 64
im_length = 64
labels = ["2x1", "cir", "rec", "squ"]  # List of strings to separate images based on classifier
model = LogisticRegression(max_iter = 1000)

# Fitting the model on training data
x_train, y_train = get_feature_and_label_arrays(path_training, labels, im_width)
model.fit(x_train, y_train)

# Obtaining test data
x_test, y_test = get_feature_and_label_arrays(path_testing, labels, im_width)

# Running test on training data to see its accuracy score
test_function(path_training, model, labels, im_width)

P_train = len(x_train[:,0])
P_test = len(x_test[:,0])
print(P_train,P_test)

C:\Users\Matt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Confusion Matrix:
[[27  0  0  0]
 [ 0 27  0  0]
 [ 0  0 27  0]
 [ 0  0  0 27]]

Accuracy Score:
1.0
108 108


In [80]:
ET = 64 # edge threshold  
PT = 0.5 # peak threshold

edge_features_train = np.empty((P_train,2))

for i in range(P_train):
    edge_features_train[i,:] = process_image(x_train[i,:],ET,PT, im_width, im_length)

plt.scatter(edge_features_train[:,0], edge_features_train[:,1], c = y_train)
plt.xlabel('pnr_v')
plt.ylabel('pnr_h')
plt.grid(1)
plt.show()
    
edge_features_test = np.empty((P_test,2))

for i in range(P_test):
    edge_features_test[i,:] = process_image(x_test[i,:],ET,PT,im_width,im_length)

# plt.scatter(edge_features_test[:,0], edge_features_test[:,1], c = y_test)
# plt.xlabel('pnr_v')
# plt.ylabel('pnr_h')
# plt.grid(1)
# plt.show()

NameError: name 'features' is not defined

In [78]:
# train logistic regression model and test it
model = LogisticRegression()
model.fit(edge_features_train,y_train)

y_pred = model.predict(edge_features_test)
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

test_function(path_testing,model,labels,im_width)

C:\Users\Matt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.16666666666666666
[[ 4  8  5 10]
 [ 8  2  8  9]
 [ 7  3  7 10]
 [10  5  7  5]]


ValueError: X has 4096 features, but LogisticRegression is expecting 3 features as input.